In [ ]:
import kagglehub


%pip install catboost -q


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
import pandas as pd


full_path = path + "/Q3_data.csv"
df = pd.read_csv(full_path)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
def check_missing_values(df):

  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])

  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

print('pre-cleaning: \n')
check_missing_values(df)


df_clean = df.dropna(axis=1)

print('\n\npost-cleaning:')
check_missing_values(df_clean)


In [ ]:
def check_duplicates(df):
  duplicates = df.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols)) # Since there are no categorical variables, no categorical encoding necessary (If we did we could just use one hot scaler.)


In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scale = scaler.fit_transform(df)

In [ ]:
import matplotlib.pyplot as plt

def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()
  plt.show()

check_target_imbalance(df, "Target") # I am fairly certain that it is imbalanced with a heavy skew towards 0.

In [ ]:
X = df_clean.drop("Target", axis=1).astype(float)
y = df_clean['Target'].astype(float)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from catboost import CatBoostClassifier
import numpy as np

sklearn_models = {
  "CatBoost": CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
}
all_results = {}

for name in sklearn_models:
  all_results[name] = {'f1': []}

skf = StratifiedKFold(n_splits=6, shuffle=True, random_state=64)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{6}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in sklearn_models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    f1 = f1_score(y_test, y_pred, zero_division=0)

    all_results[model_name]['f1'].append(f1)
print(f"  F1-Score:  {np.mean(all_results[model_name]['f1']):.4f}")

In [ ]:
#Out of Time anything that works here is fine!!

importances= {}

importances['CatBoost'] = sklearn_models['CatBoost'].feature_importances_
fig, axes = plt.subplots(1, 3, figsize=(18, 60))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# D47 as per our graph